# Committor Model: Non-Parametric Approach

Trains a non-parametric committor (`CommittorNE`) on a single long MD trajectory using RMSD-based boundary states. The model predicts the probability of reaching the folded state (low RMSD) before the unfolded state (high RMSD).

**Data:**
- `2f4kpnas380.CArmsd` — per-frame RMSD values
- `csdih` — per-frame dihedral angles (Fortran binary, 136 dihedrals)

**Boundaries:** `rmsd > 10.5` (unfolded) and `rmsd < 1.0` (folded)

## 1. Imports

In [ ]:
import numpy as np
import os
import math
import matplotlib.pyplot as plt

# Plot styling
plt.rc('font', size=10)
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)
plt.rc('legend', fontsize=12)

# Custom optimal-RC / committor package
import sys
sys.path.append('/home/staff7/fbsskr/Notebooks/optimalrcs-cupy/')
import optimalrcs, metrics, plots

## 2. Data Loading

Reads RMSD (text file, one value per line) and dihedral angles (Fortran unformatted binary) from the MD trajectory.

In [ ]:
# --- RMSD ---
with open('2f4kpnas380.CArmsd') as f:
    rmsd = np.asarray([float(l.split()[1]) for l in f], dtype=np.float32)


# --- Dihedral angles (Fortran binary) ---
def read_dih(name):
    """Read dihedral angles from a Fortran unformatted binary file."""
    from scipy.io import FortranFile
    f = FortranFile(name, 'r')
    rec = f.read_record(np.dtype('i4'))
    nsets, ndih = rec[0], rec[1]
    csdih = f.read_record(np.dtype('f4')).reshape((ndih, nsets))
    return np.transpose(csdih)  # shape: (nsets, ndih)


csdih = read_dih('csdih')
nsets, ndih = csdih.shape
print(f'Dihedrals: {csdih.shape}, RMSD: {rmsd.shape}')

## 3. Free Energy Profile (FEP)

Plots the FEP of the RMSD coordinate to visualize the two-state landscape (folded basin vs unfolded basin).

In [ ]:
fig, ax = plt.subplots(1, 1)
plots.plot_fep(ax, rmsd, xlabel='RMSD')
plt.show()

## 4. Trajectory Indices

Defines trajectory indexing for the committor model.
Here we have a single trajectory (`i_traj = 1` for all frames).

In [ ]:
i_traj = np.ones_like(rmsd)          # trajectory index (1 = single trajectory)
t_traj = np.asarray(range(len(rmsd)))  # frame index
print(f'Frames: {len(rmsd)}')

## 5. Train Committor (Non-Parametric)

Trains `CommittorNE` over 10 random seeds. Each run uses a random dihedral as the observed variable `y`, with history features.

**Parameters:**
| Parameter | Value | Description |
|-----------|-------|-------------|
| `envelope_scale` | 0.001 | RC envelope width |
| `history` | 0..10 | History lag times |
| `gamma` | 0.2 | RC regularization |
| `max_iter` | 45000 | Maximum iterations |
| `history_type` | `y(t-d),r(t-d)` | Feature types |
| `history_shift_type` | `r(t)` | Shift reference |

In [ ]:
# Configuration
optimalrcs.envelope_scale = 0.001
history = list(range(11))          # lag times 0..10
gamma = 0.2                        # regularization strength
max_iter = 45000                   # iterations per seed
n_seeds = 10                       # number of random seeds

# Feature specification
yr_type = ['y(t-d),r(t-d)']       # observed var + history features
shift_type = 'r(t)'                # shift reference

# Random dihedral as observed variable (changes each call)
def comp_y():
    return csdih[:, np.random.randint(csdih.shape[1])]


# Training loop over seeds
for seed in range(n_seeds):
    np.random.seed(seed)
    print(f'Seed {seed}')

    q = optimalrcs.CommittorNE(
        boundary0=rmsd > 10.5,   # unfolded state
        boundary1=rmsd < 1.0,    # folded state
        i_traj=i_traj,
        t_traj=t_traj,
    )

    q.fit_transform(
        comp_y,
        history_delta_t=history,
        gamma=gamma,
        max_iter=max_iter,
        history_type=yr_type,
        history_shift_type=shift_type,
    )

    # Diagnostics: FEP, observed vs predicted, log-scale
    q.plots_feps()
    q.plots_obs_pred()
    q.plots_obs_pred(log_scale=True)

## 6. Save Results

Uncomment to save the raw committor trajectory and its normalized version to disk.

In [ ]:
# Save raw committor values
# np.save('q_traj.npy', q.r_traj.numpy())

# Save normalized committor
# qn_traj = plots.transform_q2qn(q.r_traj.numpy())
# np.save('qn_traj.npy', qn_traj)